In [35]:
import numpy as np
import pandas
import matplotlib.pyplot as plt

import scipy.stats

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import sys
import requests
import importlib

def import_local_or_github(package_name, function_name=None, directory=None, giturl=None):
    # Import functions directly from github
    # Important: note that we use raw.githubusercontent.com, not github.com

    try: # to find the file locally
        if directory is not None:
            if directory not in sys.path:
                sys.path.append(directory)

        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package, function_name)
            return function
        else:
            return package

    except: # get the file from github
        if giturl is None:
            giturl = 'https://raw.githubusercontent.com/florisvb/Nonlinear_and_Data_Driven_Estimation/main/Utility/' + str(package_name) + '.py'

        r = requests.get(giturl)
        print('Fetching from: ')
        print(r)

        # Store the file to the colab working directory
        with open(package_name+'.py', 'w') as f:
            f.write(r.text)
        f.close()

        # import the function we want from that file
        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package, function_name)
            return function
        else:
            return package

#planar_drone = import_local_or_github('planar_drone', directory='../Utility')
plot_tme = import_local_or_github('plot_utility', 'plot_tme', directory='../Utility')

# Load the data

And format it so the shapes and organization of the data matches standard mathematical notation.

### Load the arrays for the kalman filter

Colab tip: You may need to download them to your machine from notebook A, and then upload them to the colab environment where you are running this notebook.

In [38]:
# read from disk
loaded_arrays = np.load('ABCDRQ.npz')
A = loaded_arrays['A']
B = loaded_arrays['B']
C = loaded_arrays['C']
D = loaded_arrays['D']
R = loaded_arrays['R']
Q = loaded_arrays['Q']

In [39]:
print('A shape: ', A.shape)
print('B shape: ', B.shape)
print('C shape: ', C.shape)
print('D shape: ', D.shape)
print('R shape: ', R.shape)
print('Q shape: ', Q.shape)

A shape:  (7, 7)
B shape:  (7, 2)
C shape:  (4, 7)
D shape:  (4, 2)
R shape:  (4, 4)
Q shape:  (7, 7)


### Load the state data and reorganize Y and U

Standard mathematical notation requires that each column of $Y$ correspond to the measurements at discrete time, i.e. $\mathbf{y}_k$, and each row of $\mathbf{y}$ correspond to each individual measurement. Similarly for $U$.

In [40]:
# read from disk
df_states = pandas.read_hdf('planar_drone_states.hdf')
df_measurements_true = pandas.read_hdf('planar_drone_measurements.hdf')
df_measurements_noisy = pandas.read_hdf('planar_drone_measurements_noisy.hdf')

In [41]:
Y = np.vstack(df_measurements_noisy[['theta', 'x', 'z', 'k']].values).T
print(Y.shape)

(4, 200)


In [42]:
U = np.vstack(df_states[['j1', 'j2']].values).T
print(U.shape)

(2, 200)


### Tip: When you index Y or U, make sure you are keeping a column vector.

In [47]:
# This is wrong
#y_0 = Y[:, 0]
print(y_0)

[[0.10971782]
 [2.20742768]
 [0.88498041]
 [1.01166974]]


In [48]:
# This is right
y_0 = Y[:, [0]]
print(y_0)

[[0.10971782]
 [2.20742768]
 [0.88498041]
 [1.01166974]]


## Tip: matrix multiplication in python

Recall that matrix shape during multiplication works like:

$[N\times M][M\times 1]=[N\times 1]$

In [49]:
# Option 1 using arrays
print('R shape: ', R.shape)
print('y_0 shape: ', y_0.shape)
print('[R x y_0] shape', (R@y_0).shape)
print('')
print('[R x y_0] = ')
R@y_0

R shape:  (4, 4)
y_0 shape:  (4, 1)
[R x y_0] shape (4, 1)

[R x y_0] = 


array([[1.09717820e-03],
       [8.82971074e-02],
       [1.41596865e-01],
       [1.01166974e-04]])

In [50]:
# Option 2 using matrices -- NOTE: np.matrix is getting deprecated, sadly, so this is not recommended
R_times_y_0 = np.matrix(R)*np.matrix(y_0)
print('[R x y_0] = ')
R_times_y_0

[R x y_0] = 


matrix([[1.09717820e-03],
        [8.82971074e-02],
        [1.41596865e-01],
        [1.01166974e-04]])

# Exercises:

1. Code a linear discrete Kalman filter that uses the matrices and time series data defined above:
    * $A, B, C, D, R, Q$ matrices defined above
    * Measurements are given by $Y$
    * Controls are given by $U$
    * Choose an initial $\mathbf{x_0}$ and $P_0$
    * Try a larger value of Q, and a smaller value of Q. For this problem, does a larger or smaller value of Q lead to more accurate estimates? Why do you think that is?

2. Compare the error covariance of your Kalman filter to the Cramer Rao Bound. Is your KF efficient? Try for different values of Q.

3. Compare the error covariance of your KF and the CRB to the actual errors in your estimate. Does the distribution of the errors match that predicted by the error covariance and the CRB? Hint: the error covariance and the CRB report variance, the square of the standard deviation.

In [74]:
# Implement the Kalman Filter
n_states = A.shape[0]
n_measurements = C.shape[0]
n_steps = Y.shape[1]

print=('[n_states]')
print=('n_measurements')
n_steps

# Initialize arrays to store estimates and error covariances
x_hat = np.zeros((n_states, n_steps))
P = np.zeros((n_states, n_states, n_steps))

# Set initial state estimate and error covariance
x_hat[:, [0]] = x_0
P[:, :, 0] = P_0

# Kalman Filter loop
for k in range(1, n_steps):
    # Prediction Step
    x_hat_minus = A @ x_hat[:, [k-1]] + B @ U[:, [k-1]]
    P_minus = A @ P[:, :, k-1] @ A.T + Q

    # Update Step
    K = P_minus @ C.T @ np.linalg.inv(C @ P_minus @ C.T + R)
    x_hat[:, [k]] = x_hat_minus + K @ (Y[:, [k]] - (C @ x_hat_minus + D @ U[:, [k]]))
    P[:, :, k] = (np.eye(n_states) - K @ C) @ P_minus

# The state estimates are now stored in x_hat, and the error covariances in P